In [1]:
# 配置依赖、数据路径、缓存文件、目标标签、模型开关和随机种子。
import ast
import hashlib
import json
import random
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import xgboost as xgb
from catboost import CatBoostClassifier
from IPython.display import display
from openai import OpenAI
from openpyxl.styles import Font
from pytorch_tabnet.tab_model import TabNetClassifier
from rtdl_revisiting_models import FTTransformer
from tab_transformer_pytorch import TabTransformer
from torch.utils.data import DataLoader, TensorDataset
from sklearn.decomposition import PCA
from threadpoolctl import threadpool_limits
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, balanced_accuracy_score, confusion_matrix, f1_score, roc_auc_score
from sklearn.preprocessing import LabelBinarizer, LabelEncoder, MultiLabelBinarizer, OneHotEncoder, StandardScaler
from sklearn.svm import SVC

warnings.filterwarnings("ignore")

DATA_FILE = Path("data/ROTD_final.xlsx")
CLINICAL_OUTCOME_SPLIT_FILE = Path("data/use_case2_clinical_outcome_prediction_split.xlsx")
RESULT_FILE = Path("results/use_case2_clinical_outcome_prediction.xlsx")
CACHE_DIR = Path("embedding_cache")
LLM_CONFIG_FILE = Path("LLM_config.json")
CACHE_DIR.mkdir(parents=True, exist_ok=True)
RESULT_FILE.parent.mkdir(parents=True, exist_ok=True)

TARGETS = [
    "Overall_outcome",
    "Visual_change",
    "Recurrence",
    "Metastasis",
    "Survival_status",
    "Adverse events",
]
LEGACY_TARGET_SHEETS = {'Overall_outcome': 'outcome_coarse_label', 'Visual_change': 'vision_label', 'Recurrence': 'recurrence_label', 'Metastasis': 'metastasis_label', 'Survival_status': 'survival_label', 'Adverse events': 'adverse_event_label'}
SETTING_NAME = "fusion"
EMBEDDING_MODEL = "text-embedding-3-large"
PCA_N_COMPONENTS = 32
RANDOM_STATE = 42
ENABLE_DEEP_MODELS = True
EMBED_BATCH_SIZE = 32
MAX_RETRIES = 5
BINARY_POSITIVE_LABELS = {
    "Recurrence": "recurrence",
    "Metastasis": "metastasis",
    "Survival_status": "dead",
    "Adverse events": "yes",
}

# 如果没有现成的 clinical outcome embedding 缓存，就从 LLM_config.json 读取配置并调用 embedding API。
def load_llm_config(config_file=LLM_CONFIG_FILE):
    if not Path(config_file).exists():
        raise FileNotFoundError(f"Missing LLM config file: {config_file}")
    with Path(config_file).open("r", encoding="utf-8-sig") as fh:
        config = json.load(fh)
    api_key = config.get("api_key") or config.get("OPENAI_API_KEY") or config.get("key")
    base_url = config.get("base_url") or config.get("OPENAI_BASE_URL") or config.get("url")
    if not api_key:
        raise ValueError(f"{config_file} must include api_key or OPENAI_API_KEY")
    return api_key, base_url

OPENAI_API_KEY, OPENAI_BASE_URL = load_llm_config()

CLINICAL_OUTCOME_CACHE_TAG = "clinical_outcome_final_pastmed_reports_tumorsize_treatment"
EMBED_MATRIX_FILE = CACHE_DIR / f"embedding_matrix_{CLINICAL_OUTCOME_CACHE_TAG}_{EMBEDDING_MODEL}.npy"
EMBED_META_FILE = CACHE_DIR / f"embedding_matrix_{CLINICAL_OUTCOME_CACHE_TAG}_{EMBEDDING_MODEL}.meta.json"
EMBED_JSON_CACHE_FILE = CACHE_DIR / f"embedding_cache_{CLINICAL_OUTCOME_CACHE_TAG}_{EMBEDDING_MODEL}.json"
LEGACY_EMBED_JSON_CACHE_FILE = CACHE_DIR / f"embedding_cache_uc2_shared2019_pastmed_reports_tumorsize_treatment_{EMBEDDING_MODEL}.json"

np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)
torch.cuda.manual_seed_all(RANDOM_STATE)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Embedding model: {EMBEDDING_MODEL}")
print(f"PCA components: {PCA_N_COMPONENTS}")
print(f"Torch device: {DEVICE}")


Embedding model: text-embedding-3-large
PCA components: 32
Torch device: cuda


In [2]:
# 定义临床结局文本拼接、embedding 缓存校验和按需生成 embedding 的工具函数。
# 将空值、占位符和无效字符串统一转换为空字符串，避免拼接到 embedding 输入中。
def normalize_text_value(value):
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return ""
    text = str(value).strip()
    return "" if text in {"", "[]", "nan", "None", "NaN"} else text


# 只使用临床结局相关文本字段构造 embedding 输入，明确排除 disease_name 和 disease_codes。
def build_clinical_outcome_text(row):
    fields = {
        "past_med_hist": normalize_text_value(row.get("Past_medical_history")) or "N/A",
        "reports": normalize_text_value(row.get("Clinical_reports")) or "N/A",
        "tumor_size": normalize_text_value(row.get("Tumor_size")) or "N/A",
        "treatment": normalize_text_value(row.get("Treatment_information")) or "N/A",
    }
    return "\n\n".join([f"{name}:\n{value}" for name, value in fields.items()])


# 为当前文本序列生成稳定哈希，用来判断 embedding cache 是否仍匹配当前输入。
def hash_texts(texts):
    hasher = hashlib.sha256()
    for text in texts:
        encoded = str(text).encode("utf-8")
        hasher.update(len(encoded).to_bytes(8, "little"))
        hasher.update(encoded)
    return hasher.hexdigest()


# 读取 JSON 缓存文件；文件不存在时返回空字典，便于首次生成缓存。
def load_json(path):
    path = Path(path)
    if not path.exists():
        return {}
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)


# 以 UTF-8 写出 JSON，保留中文和医学文本内容。
def save_json(path, payload):
    with Path(path).open("w", encoding="utf-8") as f:
        json.dump(payload, f, ensure_ascii=False, indent=2)


# 读取 clinical outcome embedding 矩阵，并用行数、模型名和文本哈希做一致性校验。
def load_embedding_matrix(matrix_file, meta_file, texts, model_name):
    matrix_file = Path(matrix_file)
    meta_file = Path(meta_file)
    if not matrix_file.exists():
        return None
    matrix = np.load(matrix_file).astype(np.float32)
    if matrix.shape[0] != len(texts):
        raise ValueError(f"Embedding row count mismatch: {matrix.shape[0]} vs {len(texts)}")
    if meta_file.exists():
        meta = load_json(meta_file)
        if meta.get("model_name") != model_name:
            raise ValueError(f"Embedding model mismatch in {meta_file}")
        if meta.get("text_hash") != hash_texts(texts):
            print(f"Embedding text hash changed in {meta_file}; rebuilding from text cache.")
            return None
    else:
        print(f"Warning: missing meta file {meta_file}; using row-count validation only.")
    print(f"Loaded embedding matrix: {matrix_file} {matrix.shape}")
    return matrix


# 保存 embedding 矩阵及其元数据，保证后续运行能复用同一批文本向量。
def save_embedding_matrix(matrix_file, meta_file, matrix, texts, model_name):
    np.save(matrix_file, matrix.astype(np.float32))
    save_json(meta_file, {
        "model_name": model_name,
        "text_count": len(texts),
        "text_hash": hash_texts(texts),
        "shape": list(matrix.shape),
        "dtype": str(matrix.dtype),
    })


# 在没有矩阵缓存时调用 text-embedding-3-large API，并逐批更新单条文本缓存以支持断点续跑。
def generate_embeddings(texts, model_name, cache_file):
    if not OPENAI_API_KEY:
        raise RuntimeError(
            f"No clinical outcome embedding matrix was found at {EMBED_MATRIX_FILE}. "
            "Set OPENAI_API_KEY in the config cell to generate embeddings."
        )

    client_kwargs = {"api_key": OPENAI_API_KEY}
    if OPENAI_BASE_URL:
        client_kwargs["base_url"] = OPENAI_BASE_URL
    client = OpenAI(**client_kwargs)

    cache = load_json(cache_file)
    if not cache:
        cache = load_json(LEGACY_EMBED_JSON_CACHE_FILE)
        if cache:
            print(f"Reusing {len(cache)} cached clinical outcome texts from {LEGACY_EMBED_JSON_CACHE_FILE}")
    vectors = [None] * len(texts)
    uncached = []
    uncached_idx = []
    for idx, text in enumerate(texts):
        if text in cache:
            vectors[idx] = np.asarray(cache[text], dtype=np.float32)
        else:
            uncached.append(text)
            uncached_idx.append(idx)
    print(f"Embedding cache hits: {len(texts) - len(uncached)}/{len(texts)}; API inputs needed: {len(uncached)}")

    for start in range(0, len(uncached), EMBED_BATCH_SIZE):
        batch_texts = uncached[start:start + EMBED_BATCH_SIZE]
        batch_indices = uncached_idx[start:start + EMBED_BATCH_SIZE]
        last_error = None
        for attempt in range(MAX_RETRIES):
            try:
                response = client.embeddings.create(model=model_name, input=batch_texts)
                break
            except Exception as exc:
                last_error = exc
                time.sleep(min(2 ** attempt, 10))
        else:
            raise RuntimeError(f"Embedding request failed: {last_error}")
        for row_idx, item in zip(batch_indices, response.data):
            vector = np.asarray(item.embedding, dtype=np.float32)
            vectors[row_idx] = vector
            cache[texts[row_idx]] = vector.tolist()
        save_json(cache_file, cache)
        print(f"Embedded {min(start + EMBED_BATCH_SIZE, len(uncached))}/{len(uncached)} uncached texts")
    if not Path(cache_file).exists():
        save_json(cache_file, cache)
    return np.vstack(vectors).astype(np.float32)

In [3]:
# 定义结构化特征工程函数，包括年龄、人口学分类变量和多标签代码展开。
# 将 Excel 中保存的列表字符串解析成 Python 列表，兼容空值和普通字符串。
def parse_list_value(value):
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return []
    text = str(value).strip()
    if text in {"", "[]", "nan", "None", "NaN"}:
        return []
    try:
        parsed = ast.literal_eval(text)
        if isinstance(parsed, (list, tuple, set)):
            return sorted({str(item).strip() for item in parsed if str(item).strip()})
    except (ValueError, SyntaxError):
        pass
    return [text]


# 创建兼容不同 scikit-learn 版本的 OneHotEncoder。
def make_one_hot_encoder():
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)


# 清洗特征名中的 XGBoost 禁用字符，并确保列名唯一。
def make_unique_feature_names(names):
    seen = {}
    unique = []
    for raw_name in names:
        name = str(raw_name)
        name = (
            name.replace("[", "(")
            .replace("]", ")")
            .replace("<", "lt_")
            .replace("\n", " ")
            .replace("\r", " ")
            .strip()
        )
        if not name:
            name = "feature"
        count = seen.get(name, 0)
        unique.append(name if count == 0 else f"{name}__{count}")
        seen[name] = count + 1
    return unique


# 将 age_y、age_m、age_d 合成为以年为单位的连续年龄特征。
def calculate_age_years(row):
    y_val = pd.to_numeric(row.get("Age_y"), errors="coerce")
    m_val = pd.to_numeric(row.get("Age_m"), errors="coerce")
    d_val = pd.to_numeric(row.get("Age_d"), errors="coerce")
    if pd.isna(y_val) and pd.isna(m_val) and pd.isna(d_val):
        return np.nan
    return (0 if pd.isna(y_val) else y_val) + (0 if pd.isna(m_val) else m_val) / 12 + (0 if pd.isna(d_val) else d_val) / 365


# 只根据训练集确定多标签特征空间，再将训练集和测试集展开为二值矩阵。
def transform_multilabel(train_series, test_series, prefix):
    train_lists = train_series.apply(parse_list_value)
    test_lists = test_series.apply(parse_list_value)
    classes = sorted({item for values in train_lists for item in values})
    if not classes:
        return np.zeros((len(train_series), 0), dtype=np.float32), np.zeros((len(test_series), 0), dtype=np.float32), []
    mlb = MultiLabelBinarizer(classes=classes)
    X_train = mlb.fit_transform(train_lists).astype(np.float32)
    X_test = mlb.transform(test_lists).astype(np.float32)
    return X_train, X_test, [f"{prefix}::{item}" for item in classes]


# 构建结构化特征矩阵；临床结局预测默认纳入 disease_codes，符合已知诊断后的临床临床结局场景。
def build_structured_features(train_df, test_df, include_disease_codes=True):
    train_df = train_df.copy()
    test_df = test_df.copy()
    train_df["age"] = train_df.apply(calculate_age_years, axis=1)
    test_df["age"] = test_df.apply(calculate_age_years, axis=1)

    imputer = SimpleImputer(strategy="median")
    age_scaler = StandardScaler()
    X_train_age = age_scaler.fit_transform(imputer.fit_transform(train_df[["age"]].values)).astype(np.float32)
    X_test_age = age_scaler.transform(imputer.transform(test_df[["age"]].values)).astype(np.float32)

    cat_cols = ["Gender", "Ethnicity", "Country"]
    train_cat = train_df[cat_cols].fillna("Unknown").astype(str)
    test_cat = test_df[cat_cols].fillna("Unknown").astype(str)
    ohe = make_one_hot_encoder()
    X_train_cat = ohe.fit_transform(train_cat).astype(np.float32)
    X_test_cat = ohe.transform(test_cat).astype(np.float32)
    feature_names = ["age"] + list(ohe.get_feature_names_out(cat_cols))

    arrays_train = [X_train_age, X_train_cat]
    arrays_test = [X_test_age, X_test_cat]
    multilabel_feature_cols = [
        ("Phenotype_ids", "hpo"),
        ("Anatomical_location_ids", "location"),
        ("Genetic_testing_method", "Genetic_testing_method"),
        ("Genotype", "Genotype"),
        ("Mutation_type", "Mutation_type"),
        ("Mutation_gene_name", "Mutation_gene_name"),
        ("HGVS variant description", "HGVS variant description"),
        ("Chromosome", "Chromosome"),
    ]
    for col, prefix in multilabel_feature_cols:
        X_train, X_test, names = transform_multilabel(
            train_df.get(col, pd.Series([np.nan] * len(train_df))),
            test_df.get(col, pd.Series([np.nan] * len(test_df))),
            prefix,
        )
        arrays_train.append(X_train)
        arrays_test.append(X_test)
        feature_names.extend(names)

    if include_disease_codes:
        X_train, X_test, names = transform_multilabel(
            train_df.get("Disease_ids", pd.Series([np.nan] * len(train_df))),
            test_df.get("Disease_ids", pd.Series([np.nan] * len(test_df))),
            "disease_code",
        )
        arrays_train.append(X_train)
        arrays_test.append(X_test)
        feature_names.extend(names)

    return np.hstack(arrays_train).astype(np.float32), np.hstack(arrays_test).astype(np.float32), feature_names


# 判断某个临床结局标签是否可用于 available-case 分析。
def is_available_label(value):
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return False
    return str(value).strip() not in {"", "nan", "None", "NaN", "[]"}


# 将标签分布压缩为 JSON 字符串，方便写入 Excel 汇总表。
def summarize_distribution(values):
    return json.dumps(pd.Series(values).astype(str).value_counts(dropna=False).to_dict(), ensure_ascii=False)


# 为单个 clinical outcome target 创建分层 train/test split，仅使用该 target 的 available cases。
def create_target_split(target_df, target):
    labels = target_df[target].astype(str).str.strip()
    split_df = pd.DataFrame({"pat_id": target_df["pat_id"].astype(str), "split": "train"})
    label_counts = labels.value_counts()
    can_stratify = len(label_counts) >= 2 and label_counts.min() >= 2
    test_size = 0.3
    rng = np.random.RandomState(RANDOM_STATE)

    if can_stratify:
        test_indices = []
        for _, group_indices in labels.groupby(labels).groups.items():
            group_indices = np.asarray(list(group_indices))
            rng.shuffle(group_indices)
            n_test = int(round(len(group_indices) * test_size))
            n_test = min(max(n_test, 1), len(group_indices) - 1)
            test_indices.extend(group_indices[:n_test].tolist())
        split_df.loc[test_indices, "split"] = "test"
    else:
        shuffled = np.arange(len(target_df))
        rng.shuffle(shuffled)
        n_test = max(1, int(round(len(shuffled) * test_size)))
        n_test = min(n_test, len(shuffled) - 1) if len(shuffled) > 1 else 0
        split_df.loc[shuffled[:n_test], "split"] = "test"

    return split_df


# 读取或生成每个 clinical outcome target 的独立 split，Excel 中每个 sheet 对应一个 target。
def load_or_create_clinical_outcome_splits(raw_data, targets, split_file=CLINICAL_OUTCOME_SPLIT_FILE):
    split_file = Path(split_file)
    split_map = {}
    changed = not split_file.exists()
    if split_file.exists():
        with pd.ExcelFile(split_file) as workbook:
            for target in targets:
                sheet = target if target in workbook.sheet_names else LEGACY_TARGET_SHEETS.get(target)
                if sheet in workbook.sheet_names:
                    split_map[target] = pd.read_excel(workbook, sheet_name=sheet, dtype={"pat_id": str, "split": str})
                    changed = changed or sheet != target
    else:
        split_file.parent.mkdir(parents=True, exist_ok=True)

    for target in targets:
        available = raw_data[target].apply(is_available_label)
        target_df = raw_data.loc[available, ["pat_id", target]].copy().reset_index(drop=True)
        if split_map.get(target) is None:
            split_map[target] = create_target_split(target_df, target)
            changed = True
        expected_ids = set(target_df["pat_id"].astype(str))
        saved_ids = set(split_map[target]["pat_id"].astype(str))
        if expected_ids != saved_ids:
            raise ValueError(f"{target}: existing patient split differs from current available cases; review before rebuilding.")
        if split_map[target]["pat_id"].duplicated().any():
            raise ValueError(f"{target}: duplicate patient IDs in split.")
        invalid = sorted(set(split_map[target]["split"]) - {"train", "test"})
        if invalid:
            raise ValueError(f"{target} split contains invalid labels: {invalid}")

    if changed:
        with pd.ExcelWriter(split_file, engine="openpyxl") as writer:
            for target in targets:
                split_map[target].to_excel(writer, sheet_name=target, index=False)
        print(f"Saved clinical outcome target splits to {split_file}")
    else:
        print(f"Loaded clinical outcome target splits from {split_file}")
    return split_map



In [4]:
# 定义传统模型、深度模型、统一概率输出和评估指标计算函数。
# 统一登记要比较的模型，后续主循环只依赖 model_registry 执行训练和评估。
def make_model_registry():
    registry = [
        ("RandomForest", "sklearn", lambda n_classes: RandomForestClassifier(n_estimators=500, max_depth=15, min_samples_leaf=2, class_weight="balanced_subsample", max_features="sqrt", random_state=RANDOM_STATE, n_jobs=-1)),
    ]
    # 根据二分类或多分类任务生成对应的 XGBoost 参数。
    def make_xgboost(n_classes):
        params = {
            "objective": "binary:logistic" if n_classes == 2 else "multi:softprob",
            "n_estimators": 500,
            "learning_rate": 0.005,
            "max_depth": 2,
            "subsample": 0.6,
            "colsample_bytree": 0.6,
            "reg_alpha": 1.0,
            "reg_lambda": 2.0,
            "random_state": RANDOM_STATE,
            "tree_method": "hist",
            "eval_metric": "logloss" if n_classes == 2 else "mlogloss",
        }
        if n_classes > 2:
            params["num_class"] = n_classes
        return xgb.XGBClassifier(**params)

    registry.append(("XGBoost", "sklearn", make_xgboost))
    registry.append(("LogisticRegression", "sklearn", lambda n_classes: LogisticRegression(max_iter=5000, C=0.1, class_weight="balanced", solver="lbfgs", random_state=RANDOM_STATE)))
    registry.append(("CatBoost", "sklearn", lambda n_classes: CatBoostClassifier(
        iterations=500, learning_rate=0.01, depth=4, l2_leaf_reg=5.0, auto_class_weights="Balanced",
        loss_function="Logloss" if n_classes == 2 else "MultiClass", random_state=RANDOM_STATE, verbose=0,
    )))
    registry.append(("SVC", "sklearn", lambda n_classes: SVC(kernel="linear", C=0.01, probability=True, class_weight="balanced", random_state=RANDOM_STATE)))
    if ENABLE_DEEP_MODELS:
        registry.extend([
            ("MLP", "deep", None),
            ("TabNet", "deep", None),
            ("TabTransformer", "deep", None),
            ("FTTransformer", "deep", None),
        ])
    return registry


# 将不同 sklearn 模型的 predict_proba 输出对齐为固定类别顺序的概率矩阵。
def normalized_scores_from_model(model, X_input, class_values):
    if hasattr(model, "predict_proba"):
        proba = np.asarray(model.predict_proba(X_input), dtype=float)
        model_classes = np.asarray(getattr(model, "classes_", class_values))
        scores = np.zeros((len(X_input), len(class_values)), dtype=float)
        for col, cls in enumerate(model_classes):
            if cls in set(class_values):
                scores[:, int(np.where(class_values == cls)[0][0])] = proba[:, col]
        sums = scores.sum(axis=1, keepdims=True)
        return np.divide(scores, sums, out=np.full_like(scores, 1 / len(class_values)), where=sums > 0)
    pred = np.asarray(model.predict(X_input), dtype=int)
    scores = np.zeros((len(X_input), len(class_values)), dtype=float)
    scores[np.arange(len(X_input)), pred] = 1.0
    return scores


# 固定 PyTorch 相关随机种子，减少深度模型重复运行时的波动。
def set_torch_seed():
    random.seed(RANDOM_STATE)
    np.random.seed(RANDOM_STATE)
    torch.manual_seed(RANDOM_STATE)
    torch.cuda.manual_seed_all(RANDOM_STATE)


# 根据训练集类别频数生成交叉熵 class weight，缓解类别不平衡。
def class_weight_tensor(y_train, n_classes):
    counts = np.bincount(y_train, minlength=n_classes).astype(np.float32)
    weights = counts.sum() / np.maximum(counts, 1.0) / n_classes
    return torch.tensor(weights, dtype=torch.float32, device=DEVICE)


# 为含 BatchNorm 的 MLP 选择 batch size，避免最后一个 batch 只有 1 个样本导致训练失败。
def make_non_singleton_loader(X_tensor, y_tensor, preferred_batch_size=32):
    n_samples = len(y_tensor)
    batch_size = min(preferred_batch_size, n_samples)
    while batch_size > 2 and n_samples % batch_size == 1:
        batch_size -= 1
    return DataLoader(TensorDataset(X_tensor, y_tensor), batch_size=batch_size, shuffle=True)


# 定义与诊断任务一致的多层感知机，用于单任务临床结局分类。
class ClinicalOutcomePredictionMLP(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(in_dim, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.6),
            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.6),
            nn.Linear(256, 128),
            nn.LayerNorm(128),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(128, out_dim),
        )

    def forward(self, x):
        return self.layers(x)


# 训练 MLP，并返回测试集各类别概率。
def train_mlp(X_train, y_train, X_test, n_classes):
    set_torch_seed()
    Xtr = torch.tensor(X_train, dtype=torch.float32, device=DEVICE)
    Xte = torch.tensor(X_test, dtype=torch.float32, device=DEVICE)
    ytr = torch.tensor(y_train, dtype=torch.long, device=DEVICE)
    loader = make_non_singleton_loader(Xtr, ytr, preferred_batch_size=32)
    model = ClinicalOutcomePredictionMLP(X_train.shape[1], n_classes).to(DEVICE)
    opt = optim.AdamW(model.parameters(), lr=5e-4, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=100)
    loss_fn = nn.CrossEntropyLoss(weight=class_weight_tensor(y_train, n_classes))
    for _ in range(300):
        model.train()
        for xb, yb in loader:
            opt.zero_grad()
            loss = loss_fn(model(xb), yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
        scheduler.step()
    model.eval()
    with torch.no_grad():
        return torch.softmax(model(Xte), dim=1).cpu().numpy()


# 训练 TabNet 单任务分类器，并返回测试集各类别概率。
def train_tabnet(X_train, y_train, X_test, n_classes):
    clf = TabNetClassifier(n_d=16, n_a=16, n_steps=3, gamma=1.2, lambda_sparse=5e-4, seed=RANDOM_STATE, verbose=0)
    clf.fit(X_train, y_train, max_epochs=100, batch_size=64, virtual_batch_size=16)
    return np.asarray(clf.predict_proba(X_test), dtype=float)


# 训练 FTTransformer，将全部 fusion 特征作为连续变量输入。
def train_ft_transformer(X_train, y_train, X_test, n_classes):
    set_torch_seed()
    Xtr = torch.tensor(X_train, dtype=torch.float32, device=DEVICE)
    Xte = torch.tensor(X_test, dtype=torch.float32, device=DEVICE)
    ytr = torch.tensor(y_train, dtype=torch.long, device=DEVICE)
    loader = DataLoader(TensorDataset(Xtr, ytr), batch_size=32, shuffle=True)
    model = FTTransformer(n_cont_features=X_train.shape[1], cat_cardinalities=[], d_out=n_classes, n_blocks=1, d_block=16, attention_n_heads=4, ffn_d_hidden_multiplier=2, ffn_activation="ReGLU", attention_dropout=0.1, ffn_dropout=0.1, residual_dropout=0.0).to(DEVICE)
    opt = optim.AdamW(model.parameters(), lr=3e-3, weight_decay=2e-3)
    loss_fn = nn.CrossEntropyLoss(weight=class_weight_tensor(y_train, n_classes))
    for _ in range(120):
        model.train()
        for xb, yb in loader:
            opt.zero_grad()
            loss = loss_fn(model(xb, None), yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
    model.eval()
    with torch.no_grad():
        return torch.softmax(model(Xte, None), dim=1).cpu().numpy()


# 用一个占位 categorical token 包装 TabTransformer，使其可处理当前连续 fusion 特征。
class ContinuousTabTransformer(nn.Module):
    # 初始化 TabTransformer，并声明全部输入特征为连续变量。
    def __init__(self, num_continuous, num_classes):
        super().__init__()
        self.model = TabTransformer(categories=(1,), num_continuous=num_continuous, dim=16, depth=2, heads=2, dim_out=num_classes, mlp_hidden_mults=(2,), mlp_act=nn.ReLU(), attn_dropout=0.3, ff_dropout=0.3)

    # 前向传播时补一个虚拟分类特征，满足 TabTransformer 的输入接口。
    def forward(self, x_cont):
        x_cat = torch.zeros((x_cont.shape[0], 1), dtype=torch.long, device=x_cont.device)
        return self.model(x_cat, x_cont)


# 训练 TabTransformer 包装模型，并输出测试集类别概率。
def train_tab_transformer(X_train, y_train, X_test, n_classes):
    set_torch_seed()
    Xtr = torch.tensor(X_train, dtype=torch.float32, device=DEVICE)
    Xte = torch.tensor(X_test, dtype=torch.float32, device=DEVICE)
    ytr = torch.tensor(y_train, dtype=torch.long, device=DEVICE)
    loader = DataLoader(TensorDataset(Xtr, ytr), batch_size=16, shuffle=True)
    model = ContinuousTabTransformer(X_train.shape[1], n_classes).to(DEVICE)
    opt = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
    loss_fn = nn.CrossEntropyLoss(weight=class_weight_tensor(y_train, n_classes))
    for _ in range(100):
        model.train()
        for xb, yb in loader:
            opt.zero_grad()
            loss = loss_fn(model(xb), yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
    model.eval()
    with torch.no_grad():
        return torch.softmax(model(Xte), dim=1).cpu().numpy()


# 根据模型类型分发到 sklearn 或深度模型训练函数，并统一返回概率矩阵。
def fit_and_score(model_name, model_kind, make_model, X_train_df, X_test_df, y_train, n_classes):
    if model_kind == "sklearn":
        model = make_model(n_classes)
        model.fit(X_train_df, y_train)
        return normalized_scores_from_model(model, X_test_df, np.arange(n_classes))
    X_train = X_train_df.to_numpy(np.float32)
    X_test = X_test_df.to_numpy(np.float32)
    if model_name == "MLP":
        return train_mlp(X_train, y_train, X_test, n_classes)
    if model_name == "TabNet":
        return train_tabnet(X_train, y_train, X_test, n_classes)
    if model_name == "TabTransformer":
        return train_tab_transformer(X_train, y_train, X_test, n_classes)
    if model_name == "FTTransformer":
        return train_ft_transformer(X_train, y_train, X_test, n_classes)
    raise ValueError(model_name)


# 根据概率矩阵计算二分类和多分类通用指标，同时返回预测类别用于混淆矩阵。
def evaluate_scores(y_true, scores, label_names, positive_label=None):
    y_pred = scores.argmax(axis=1)
    n_classes = len(label_names)
    if n_classes == 2:
        if positive_label not in label_names:
            raise ValueError(f"A binary task must specify its positive label from {label_names}.")
        positive_idx = label_names.index(positive_label)
    metrics = {
        "auroc": np.nan,
        "auprc": np.nan,
        "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "macro_ovr_auroc": np.nan,
        "notes": "",
    }
    try:
        if n_classes == 2:
            y_positive = (y_true == positive_idx).astype(int)
            metrics["auroc"] = roc_auc_score(y_positive, scores[:, positive_idx])
            metrics["macro_ovr_auroc"] = metrics["auroc"]
            metrics["auprc"] = average_precision_score(y_positive, scores[:, positive_idx])
        else:
            y_bin = LabelBinarizer().fit_transform(y_true)
            metrics["macro_ovr_auroc"] = roc_auc_score(y_bin, scores, average="macro", multi_class="ovr")
            metrics["auprc"] = average_precision_score(y_bin, scores, average="macro")
    except ValueError as exc:
        metrics["notes"] = f"Metric warning: {exc}"
    return metrics, y_pred

In [5]:
# 读取原始数据，生成或加载每个临床结局标签独立的 split，并加载或生成临床结局专用 embedding 矩阵。
raw_data = pd.read_excel(DATA_FILE)
required_cols = [
    "pat_id", "Age_y", "Age_m", "Age_d", "Gender", "Ethnicity", "Country",
    "Phenotype_ids", "Anatomical_location_ids", "Disease_ids",
    "Genetic_testing_method", "Genotype", "Mutation_type", "Mutation_gene_name",
    "HGVS variant description", "Chromosome", "Past_medical_history",
    "Clinical_reports", "Tumor_size", "Treatment_information", *TARGETS,
]
missing_cols = [col for col in required_cols if col not in raw_data.columns]
if missing_cols:
    raise ValueError(f"Missing required source columns: {missing_cols}")
raw_data["pat_id"] = raw_data["pat_id"].astype(str)
if raw_data["pat_id"].duplicated().any():
    raise ValueError("Duplicate pat_id values found in data file.")

clinical_outcome_split_map = load_or_create_clinical_outcome_splits(raw_data, TARGETS, CLINICAL_OUTCOME_SPLIT_FILE)
data = raw_data.reset_index(drop=True).copy()
data["clinical_outcome_input_text"] = data.apply(build_clinical_outcome_text, axis=1)
texts_for_embedding = data["clinical_outcome_input_text"].tolist()
embedding_matrix = load_embedding_matrix(EMBED_MATRIX_FILE, EMBED_META_FILE, texts_for_embedding, EMBEDDING_MODEL)
if embedding_matrix is None:
    print(f"No cached clinical outcome embedding matrix found at {EMBED_MATRIX_FILE}.")
    print(f"Generating clinical outcome embeddings with {EMBEDDING_MODEL} via OpenAI API...")
    embedding_matrix = generate_embeddings(texts_for_embedding, EMBEDDING_MODEL, EMBED_JSON_CACHE_FILE)
    save_embedding_matrix(EMBED_MATRIX_FILE, EMBED_META_FILE, embedding_matrix, texts_for_embedding, EMBEDDING_MODEL)

print(f"Rows in source data: {len(data)}")
print(f"Embedding shape: {embedding_matrix.shape}")


Loaded clinical outcome target splits from data\use_case2_clinical_outcome_prediction_split.xlsx
Loaded embedding matrix: embedding_cache\embedding_matrix_clinical_outcome_final_pastmed_reports_tumorsize_treatment_text-embedding-3-large.npy (2134, 3072)
Rows in source data: 2134
Embedding shape: (2134, 3072)


In [6]:
# 按每个临床结局标签的 available cases 运行 early-fusion benchmark。
model_registry = make_model_registry()
print("Enabled models:", [name for name, _, _ in model_registry])

summary_rows = []
metric_rows = []
confusion_rows = []

for target in TARGETS:
    valid_mask = data[target].apply(is_available_label)
    target_df = data.loc[valid_mask].copy()
    split_df = clinical_outcome_split_map[target].set_index("pat_id")["split"]
    target_df["split"] = target_df["pat_id"].astype(str).map(split_df)
    if target_df["split"].isna().any():
        raise ValueError(f"{target} has patients missing from clinical outcome split.")
    full_indices = target_df.index.to_numpy()
    labels_raw = target_df[target].astype(str).str.strip()
    train_mask = target_df["split"].eq("train").to_numpy()
    test_mask = target_df["split"].eq("test").to_numpy()
    train_raw = labels_raw.iloc[train_mask]
    test_raw = labels_raw.iloc[test_mask]
    all_labels = sorted(labels_raw.unique().tolist())
    target_type = "binary" if len(all_labels) == 2 else "multiclass"

    summary_rows.append({
        "target": target,
        "target_type": target_type,
        "positive_label": BINARY_POSITIVE_LABELS.get(target, ""),
        "n_available": int(valid_mask.sum()),
        "missing_rate": float(1 - valid_mask.mean()),
        "n_train": int(train_mask.sum()),
        "n_test": int(test_mask.sum()),
        "class_distribution_train": summarize_distribution(train_raw),
        "class_distribution_test": summarize_distribution(test_raw),
    })

    if len(all_labels) < 2 or train_mask.sum() == 0 or test_mask.sum() == 0:
        print(f"Skipping {target}: insufficient available labels.")
        continue

    encoder = LabelEncoder().fit(all_labels)
    y_train = encoder.transform(train_raw)
    y_test = encoder.transform(test_raw)
    label_names = encoder.classes_.tolist()
    n_classes = len(label_names)
    if (np.bincount(y_train, minlength=n_classes) == 0).any() or (np.bincount(y_test, minlength=n_classes) == 0).any():
        print(f"Skipping {target}: at least one class is absent from train or test.")
        continue

    setting = SETTING_NAME
    train_df = target_df.loc[train_mask].copy()
    test_df = target_df.loc[test_mask].copy()

    X_train_struct, X_test_struct, feature_names = build_structured_features(train_df, test_df)
    X_train_embed = embedding_matrix[full_indices[train_mask]]
    X_test_embed = embedding_matrix[full_indices[test_mask]]

    embed_scaler = StandardScaler()
    X_train_embed = embed_scaler.fit_transform(X_train_embed)
    X_test_embed = embed_scaler.transform(X_test_embed)
    used_pca = min(PCA_N_COMPONENTS, X_train_embed.shape[0] - 1, X_train_embed.shape[1])
    pca = PCA(n_components=used_pca, random_state=RANDOM_STATE)
    # Fixed BLAS threads preserve the PCA inputs of the 2026-10-28 benchmark.
    with threadpool_limits(limits=1, user_api="blas"):
        X_train_pca = pca.fit_transform(X_train_embed).astype(np.float32)
        X_test_pca = pca.transform(X_test_embed).astype(np.float32)

    X_train = np.hstack([X_train_struct, X_train_pca]).astype(np.float32)
    X_test = np.hstack([X_test_struct, X_test_pca]).astype(np.float32)
    all_feature_names = make_unique_feature_names(feature_names + [f"embedding_pc{i + 1}" for i in range(used_pca)])
    X_train_df = pd.DataFrame(X_train, columns=all_feature_names)
    X_test_df = pd.DataFrame(X_test, columns=all_feature_names)
    base_notes = [] if used_pca == PCA_N_COMPONENTS else [f"PCA reduced from {PCA_N_COMPONENTS} to {used_pca}"]

    print(f"\nTarget={target} | Setting={setting} | train={len(y_train)} test={len(y_test)} features={X_train.shape[1]}")
    for model_name, model_kind, make_model in model_registry:
        model_notes = base_notes.copy()
        try:
            print(f"Training {model_name}...")
            scores = fit_and_score(model_name, model_kind, make_model, X_train_df, X_test_df, y_train, n_classes)
            scores = np.asarray(scores, dtype=float)
            if scores.shape != (len(y_test), n_classes):
                raise ValueError(f"Score shape {scores.shape} does not match {(len(y_test), n_classes)}")
            metrics, y_pred = evaluate_scores(y_test, scores, label_names, BINARY_POSITIVE_LABELS.get(target))
            if metrics["notes"]:
                model_notes.append(metrics["notes"])
            metric_rows.append({
                "target": target,
                "setting": setting,
                "model": model_name,
                "pca_components": int(used_pca),
                "pca_explained_variance": float(pca.explained_variance_ratio_.sum()),
                "auroc": metrics["auroc"],
                "auprc": metrics["auprc"],
                "macro_f1": metrics["macro_f1"],
                "balanced_accuracy": metrics["balanced_accuracy"],
                "macro_ovr_auroc": metrics["macro_ovr_auroc"],
                "notes": "; ".join(model_notes),
            })

            cm = confusion_matrix(y_test, y_pred, labels=np.arange(n_classes))
            for true_idx, true_label in enumerate(label_names):
                for pred_idx, pred_label in enumerate(label_names):
                    confusion_rows.append({
                        "target": target,
                        "setting": setting,
                        "model": model_name,
                        "true_label": true_label,
                        "pred_label": pred_label,
                        "count": int(cm[true_idx, pred_idx]),
                    })
        except Exception as exc:
            metric_rows.append({
                "target": target,
                "setting": setting,
                "model": model_name,
                "pca_components": int(used_pca),
                "pca_explained_variance": float(pca.explained_variance_ratio_.sum()),
                "auroc": np.nan,
                "auprc": np.nan,
                "macro_f1": np.nan,
                "balanced_accuracy": np.nan,
                "macro_ovr_auroc": np.nan,
                "notes": "; ".join(model_notes + [f"Model failed: {type(exc).__name__}: {exc}"]),
            })
            print(f"{model_name} failed: {exc}")

summary_df = pd.DataFrame(summary_rows)
metrics_df = pd.DataFrame(metric_rows)
confusion_df = pd.DataFrame(confusion_rows)
print("Benchmark complete.")
display(metrics_df.head())

Enabled models: ['RandomForest', 'XGBoost', 'LogisticRegression', 'CatBoost', 'SVC', 'MLP', 'TabNet', 'TabTransformer', 'FTTransformer']



Target=Overall_outcome | Setting=fusion | train=1259 test=539 features=2384
Training RandomForest...


Training XGBoost...


Training LogisticRegression...


Training CatBoost...


Training SVC...


... (39 outputs left out)

Training TabTransformer...


Training FTTransformer...


Benchmark complete.


,target,setting,model,pca_components,pca_explained_variance,auroc,auprc,macro_f1,balanced_accuracy,macro_ovr_auroc,notes
0,Overall_outcome,fusion,RandomForest,32,0.60938,NaN,0.606600,0.575179,0.532892,0.769977,
1,Overall_outcome,fusion,XGBoost,32,0.60938,NaN,0.558126,0.347729,0.372222,0.722037,
2,Overall_outcome,fusion,LogisticRegression,32,0.60938,NaN,0.595014,0.562440,0.619356,0.759902,
3,Overall_outcome,fusion,CatBoost,32,0.60938,NaN,0.579317,0.517905,0.574603,0.729845,
4,Overall_outcome,fusion,SVC,32,0.60938,NaN,0.536944,0.323995,0.355732,0.720227,


In [7]:
# 汇总最佳模型、运行配置，并写出单个 Excel 结果工作簿。
# 按任务类型选择主指标：二分类使用 AUROC，多分类使用 macro-F1。
PAPER_MODEL_ORDER = [
    "RandomForest",
    "XGBoost",
    "LogisticRegression",
    "CatBoost",
    "SVC",
    "MLP",
    "TabNet",
    "TabTransformer",
    "FTTransformer",
]
PAPER_PERCENT_COLS = ["auroc", "auprc", "macro_f1", "balanced_accuracy", "macro_ovr_auroc", "primary_metric_value"]

# 将临床结局结果整理为论文表格格式：只保留指定模型顺序，并把核心指标转为 xx.xx%。
def format_percent_value(value):
    if value is None or pd.isna(value):
        return ""
    if isinstance(value, str):
        text = value.strip()
        if not text or text.lower() in {"nan", "none", "na", "n/a"}:
            return ""
        if text.endswith("%"):
            return text
        value = float(text)
    return f"{float(value):.2%}"

def make_clinical_outcome_paper_ready(df):
    df = df.copy()
    if "model" in df.columns:
        order_map = {name: idx for idx, name in enumerate(PAPER_MODEL_ORDER)}
        df = df[df["model"].isin(PAPER_MODEL_ORDER)].copy()
        df["_model_order"] = df["model"].map(order_map)
        sort_cols = [col for col in ["target", "setting", "_model_order"] if col in df.columns]
        df = df.sort_values(sort_cols).drop(columns=["_model_order"]).reset_index(drop=True)
    for col in PAPER_PERCENT_COLS:
        if col in df.columns:
            df[col] = df[col].map(format_percent_value)
    return df

def choose_best_models(metrics_df, summary_df):
    if metrics_df.empty:
        return pd.DataFrame()
    target_type_map = summary_df.set_index("target")["target_type"].to_dict()
    rows = []
    for target, group in metrics_df.groupby("target"):
        target_type = target_type_map.get(target, "multiclass")
        primary_metric = "auroc" if target_type == "binary" else "macro_f1"
        values = pd.to_numeric(group[primary_metric], errors="coerce")
        if not values.notna().any():
            primary_metric = "macro_f1"
            values = pd.to_numeric(group[primary_metric], errors="coerce")
        if values.notna().any():
            best_idx = values.idxmax()
            row = group.loc[best_idx].to_dict()
            row["target_type"] = target_type
            row["primary_metric"] = primary_metric
            row["primary_metric_value"] = row[primary_metric]
            rows.append(row)
    return pd.DataFrame(rows)


if not metrics_df.empty and "model" in metrics_df.columns:
    metrics_df = metrics_df[metrics_df["model"].isin(PAPER_MODEL_ORDER)].copy()

best_models_df = choose_best_models(metrics_df, summary_df)
metrics_df_paper = make_clinical_outcome_paper_ready(metrics_df)
best_models_df_paper = make_clinical_outcome_paper_ready(best_models_df)
confusion_df_paper = make_clinical_outcome_paper_ready(confusion_df)
run_config_df = pd.DataFrame([
    {"parameter": "data_file", "value": str(DATA_FILE)},
    {"parameter": "clinical_outcome_split_file", "value": str(CLINICAL_OUTCOME_SPLIT_FILE)},
    {"parameter": "result_file", "value": str(RESULT_FILE)},
    {"parameter": "embedding_model", "value": EMBEDDING_MODEL},
    {"parameter": "embedding_matrix_file", "value": str(EMBED_MATRIX_FILE)},
    {"parameter": "embedding_text_fields", "value": "Past_medical_history; Clinical_reports; Tumor_size; Treatment_information"},
    {"parameter": "excluded_from_embedding", "value": "disease_name; disease_codes"},
    {"parameter": "pca_requested_components", "value": PCA_N_COMPONENTS},
    {"parameter": "setting", "value": SETTING_NAME},
    {"parameter": "disease_codes_in_structured_features", "value": True},
    {"parameter": "binary_positive_labels", "value": json.dumps(BINARY_POSITIVE_LABELS, ensure_ascii=False)},
    {"parameter": "models_enabled", "value": "; ".join(PAPER_MODEL_ORDER)},
    {"parameter": "deep_models_enabled", "value": ENABLE_DEEP_MODELS},
    {"parameter": "random_state", "value": RANDOM_STATE},
    {"parameter": "prediction_output", "value": "not saved by default"},
])

with pd.ExcelWriter(RESULT_FILE, engine="openpyxl") as writer:
    summary_df.to_excel(writer, sheet_name="Task_summary", index=False)
    metrics_df_paper.to_excel(writer, sheet_name="All_results", index=False)
    best_models_df_paper.to_excel(writer, sheet_name="Best_models", index=False)
    confusion_df_paper.to_excel(writer, sheet_name="Confusion_matrices", index=False)
    run_config_df.to_excel(writer, sheet_name="Run_config", index=False)

    workbook = writer.book
    bold_font = Font(bold=True)
    for sheet_name in workbook.sheetnames:
        sheet = workbook[sheet_name]
        sheet.freeze_panes = "A2"
        sheet.auto_filter.ref = sheet.dimensions
        for cell in sheet[1]:
            cell.font = bold_font

print(f"Saved benchmark workbook to {RESULT_FILE}")

Saved benchmark workbook to results\use_case2_clinical_outcome_prediction.xlsx
